In [2]:
import duckdb

from azdata.paths import RAW_DIR

In [3]:
# General information: data range, null counts
# Worldbank

result = duckdb.sql(f"""
    SELECT
        MIN(year) as first_year,
        MAX(year) as last_year,
        COUNT(*) FILTER (WHERE gdp IS NULL) as gdp_nulls,
        COUNT(*) FILTER (WHERE inflation IS NULL) as inflation_nulls,
    FROM '{RAW_DIR / "worldbank.csv"}'
""").df()

result

,first_year,last_year,gdp_nulls,inflation_nulls
0,1960,2025,31,33


In [5]:
# Fred

result = duckdb.sql(f"""
    SELECT
        MIN(year) as first_year,
        MAX(year) as last_year,
        COUNT(*) FILTER (WHERE oil_price IS NULL) as oil_price_nulls,
    FROM '{RAW_DIR / "fred.csv"}'
""").df()

result

,first_year,last_year,oil_price_nulls
0,1987-01-01,2026-01-01,2


In [ ]:
# Years with high inflation rate

result = duckdb.sql(f"""
    SELECT
        year,
        inflation,
    FROM '{RAW_DIR / "worldbank.csv"}'
    WHERE inflation > 10
""").df()

result

,year,inflation
0,1993,1128.000024
1,1994,1662.215949
2,1995,411.759642
3,1996,19.794803
4,2007,16.699755
5,2008,20.849087
6,2016,12.443375
7,2017,12.935918
8,2022,13.852259


In [ ]:
# Oil price categories

result = duckdb.sql(f"""
    SELECT
        year,
        CASE
            WHEN oil_price > 80 THEN 'High'
            WHEN oil_price BETWEEN 40 and 80 THEN 'Medium'
            WHEN oil_price < 40 THEN 'Low'
        END AS oil_price_category
    FROM '{RAW_DIR / "fred.csv"}'
""").df()

result

,year,oil_price_category
0,1987-01-01,NaN
1,1988-01-01,Low
2,1989-01-01,Low
3,1990-01-01,Low
4,1991-01-01,Low
5,1992-01-01,Low
6,1993-01-01,Low
7,1994-01-01,Low
8,1995-01-01,Low
9,1996-01-01,Low


In [34]:
result = duckdb.sql(f"""
    SELECT *
    FROM '{RAW_DIR / "fred.csv"}' f
    INNER JOIN '{RAW_DIR / "worldbank.csv"}' w
        ON EXTRACT(YEAR FROM f.year)::INTEGER = w.year
""").df()

result

,year,oil_price,year_1,gdp,unemployment_rate,exports_pct_gdp,gdp_growth,inflation,current_account,imports_pct_gdp
0,1987-01-01,NaN,1987,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1988-01-01,14.91,1988,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1989-01-01,18.23,1989,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1990-01-01,23.76,1990,8.884848e+09,NaN,43.860846,NaN,NaN,NaN,39.222374
4,1991-01-01,20.04,1991,5.344000e+09,0.900,45.658683,-0.700000,NaN,NaN,41.205090
5,1992-01-01,19.32,1992,4.446587e+08,1.800,86.203606,-22.600000,-10.630097,NaN,54.596378
6,1993-01-01,17.01,1993,1.570393e+09,4.500,57.462420,-23.099999,1128.000024,NaN,76.028025
7,1994-01-01,15.86,1994,1.193141e+09,6.300,24.721110,-19.700001,1662.215949,NaN,30.628209
8,1995-01-01,17.02,1995,2.417331e+09,7.200,32.486644,-11.799999,411.759642,-4.006500e+08,53.407067
9,1996-01-01,20.64,1996,3.176507e+09,8.100,24.928639,1.299999,19.794803,-9.311820e+08,55.921101
